# A Course Schedule Conflict Checker and Exam-aware Study Planner for College Students

## Project Introduction

College students often need to manage course schedules, assignments, exams, and personal study time at the same time. It can be difficult to notice course conflicts or plan exam review sessions early.

This project uses Python to build a simple prototype. The project reads course data, analyzes course time distribution, checks possible schedule conflicts, and creates a basic exam review plan.

This is not a complete scheduling app. It is a small data science project that applies Python, Pandas, and Matplotlib to a real student problem.

## Objective

The goal of this project is to create a simple tool for college students to organize course schedules and exam preparation.

The project has four main objectives:

1. Read and clean course data from an Excel file.
2. Analyze course distribution by weekday and course type.
3. Check whether selected courses have schedule conflicts.
4. Generate a seven-day exam review plan based on exam dates.

## Dataset Description

The dataset is a course offering file exported from the university course system. It includes course codes, course names, credits, instructors, course times, enrollment limits, and enrollment numbers.

The most important column in this project is the course time column, because it contains the weekday, class periods, campus, and classroom information.

For example:

- 一 10-B 和平 教403
- 四 9-10 和平 地理系電腦教室
- 一 7-9 和平 正205

These values are used to extract the weekday and class periods for schedule conflict checking.

## Pseudocode

1. Read the course data from an Excel file.
2. Select useful columns such as course code, course name, teacher, course time, credits, and enrollment.
3. Remove rows without course names or course time.
4. Extract the weekday and class periods from the course time column.
5. Count the number of courses by weekday.
6. Visualize course distribution using a bar chart.
7. Count required and elective courses.
8. Let the user manually enter selected course codes.
9. Compare selected courses and check if any two courses overlap.
10. Create exam dates manually.
11. Generate one review task per day for the seven days before each exam.
12. Calculate daily planned study hours.
13. Visualize daily study workload.
14. Export the final study plan as a CSV file.

## Algorithm

The conflict checking algorithm compares every pair of selected courses. If two courses are on the same weekday and their class periods overlap, the system records them as a conflict.

The study planning algorithm uses exam dates as input. For each exam, the system creates one review task per day during the seven days before the exam. Then, it calculates the total planned study hours for each date.

In [2]:
!pip install xlrd

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import re
from datetime import timedelta

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
file_name = list(uploaded.keys())[0]

course_data = pd.read_excel(file_name)

course_data.head()

In [ ]:
course_data.shape

In [ ]:
course_data.columns

In [ ]:
useful_columns = [
    "開課序號",
    "開課代碼",
    "系所",
    "中文課程名稱",
    "英文課程名稱",
    "學分",
    "必/選",
    "教師",
    "地點時間",
    "限修人數",
    "選修人數"
]

courses = course_data[useful_columns].copy()

courses.head()

In [ ]:
courses = courses.dropna(subset=["中文課程名稱", "地點時間"])

courses.head()

In [ ]:
courses["限修人數"] = pd.to_numeric(courses["限修人數"], errors="coerce")
courses["選修人數"] = pd.to_numeric(courses["選修人數"], errors="coerce")
courses["學分"] = pd.to_numeric(courses["學分"], errors="coerce")

courses.head()

In [ ]:
day_map = {
    "一": "Monday",
    "二": "Tuesday",
    "三": "Wednesday",
    "四": "Thursday",
    "五": "Friday",
    "六": "Saturday",
    "日": "Sunday"
}

period_order = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "10": 10,
    "A": 11,
    "B": 12,
    "C": 13,
    "D": 14
}

def parse_time(text):
    text = str(text)
    pattern = r"([一二三四五六日])\s+([0-9A-D]+)-([0-9A-D]+)"
    result = re.search(pattern, text)

    if result:
        day = result.group(1)
        start_period = result.group(2)
        end_period = result.group(3)

        return pd.Series([
            day_map.get(day),
            start_period,
            end_period,
            period_order.get(start_period),
            period_order.get(end_period)
        ])

    return pd.Series([None, None, None, None, None])

courses[["day", "start_period", "end_period", "start_order", "end_order"]] = courses["地點時間"].apply(parse_time)

courses.head()

In [ ]:
courses_clean = courses.dropna(subset=["day", "start_order", "end_order"]).copy()

courses_clean.head()

In [ ]:
courses_clean = courses.dropna(subset=["day", "start_order", "end_order"]).copy()

courses_clean.head()

In [ ]:
courses_by_day = courses_clean.groupby("day", observed=False)["中文課程名稱"].count().reset_index()

courses_by_day.columns = ["day", "number_of_courses"]

courses_by_day

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(courses_by_day["day"].astype(str), courses_by_day["number_of_courses"])
plt.xlabel("Day")
plt.ylabel("Number of Courses")
plt.title("Number of Courses by Weekday")
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(courses_by_day["day"].astype(str), courses_by_day["number_of_courses"])
plt.xlabel("Day")
plt.ylabel("Number of Courses")
plt.title("Number of Courses by Weekday")
plt.xticks(rotation=45)
plt.show()

In [ ]:
course_type_summary = courses_clean.groupby("必/選")["中文課程名稱"].count().reset_index()

course_type_summary.columns = ["course_type", "number_of_courses"]

course_type_summary

In [ ]:
course_type_summary = courses_clean.groupby("必/選")["中文課程名稱"].count().reset_index()

course_type_summary.columns = ["course_type", "number_of_courses"]

course_type_summary

In [ ]:
courses_clean["capacity_rate"] = courses_clean["選修人數"] / courses_clean["限修人數"]

capacity_table = courses_clean[[
    "開課代碼",
    "中文課程名稱",
    "教師",
    "限修人數",
    "選修人數",
    "capacity_rate"
]].sort_values("capacity_rate", ascending=False)

capacity_table.head(10)

In [ ]:
selected_course_codes = [
    courses_clean.iloc[0]["開課代碼"],
    courses_clean.iloc[1]["開課代碼"],
    courses_clean.iloc[2]["開課代碼"]
]

my_courses = courses_clean[courses_clean["開課代碼"].isin(selected_course_codes)].copy()

my_courses[[
    "開課代碼",
    "中文課程名稱",
    "教師",
    "地點時間",
    "day",
    "start_period",
    "end_period"
]]

In [ ]:
conflicts = []

for i in range(len(my_courses)):
    for j in range(i + 1, len(my_courses)):
        course_1 = my_courses.iloc[i]
        course_2 = my_courses.iloc[j]

        same_day = course_1["day"] == course_2["day"]
        overlap = course_1["start_order"] <= course_2["end_order"] and course_2["start_order"] <= course_1["end_order"]

        if same_day and overlap:
            conflicts.append({
                "course_1": course_1["中文課程名稱"],
                "course_2": course_2["中文課程名稱"],
                "day": str(course_1["day"]),
                "time_1": course_1["地點時間"],
                "time_2": course_2["地點時間"]
            })

conflict_table = pd.DataFrame(conflicts)

conflict_table

In [ ]:
conflicts = []

for i in range(len(my_courses)):
    for j in range(i + 1, len(my_courses)):
        course_1 = my_courses.iloc[i]
        course_2 = my_courses.iloc[j]

        same_day = course_1["day"] == course_2["day"]
        overlap = course_1["start_order"] <= course_2["end_order"] and course_2["start_order"] <= course_1["end_order"]

        if same_day and overlap:
            conflicts.append({
                "course_1": course_1["中文課程名稱"],
                "course_2": course_2["中文課程名稱"],
                "day": str(course_1["day"]),
                "time_1": course_1["地點時間"],
                "time_2": course_2["地點時間"]
            })

conflict_table = pd.DataFrame(conflicts)

conflict_table

In [ ]:
exam_list = []

number_of_exams = int(input("How many exams do you want to enter? "))

for i in range(number_of_exams):
    subject = input("Enter subject name: ")
    exam_date = input("Enter exam date (YYYY-MM-DD): ")
    importance = int(input("Enter importance from 1 to 5: "))

    exam_list.append({
        "subject": subject,
        "exam_date": exam_date,
        "importance": importance
    })

exams = pd.DataFrame(exam_list)

exams["exam_date"] = pd.to_datetime(exams["exam_date"])

exams

In [ ]:
study_plan = []

for index, row in exams.iterrows():
    subject = row["subject"]
    exam_date = row["exam_date"]
    importance = row["importance"]

    for days_before in range(7, 0, -1):
        study_date = exam_date - timedelta(days=days_before)

        study_plan.append({
            "date": study_date,
            "task": "Review " + subject,
            "type": "Exam Review",
            "subject": subject,
            "hours": 1,
            "importance": importance
        })

study_plan = pd.DataFrame(study_plan)

study_plan

In [ ]:
study_plan = []

for index, row in exams.iterrows():
    subject = row["subject"]
    exam_date = row["exam_date"]
    importance = row["importance"]

    for days_before in range(7, 0, -1):
        study_date = exam_date - timedelta(days=days_before)

        study_plan.append({
            "date": study_date,
            "task": "Review " + subject,
            "type": "Exam Review",
            "subject": subject,
            "hours": 1,
            "importance": importance
        })

study_plan = pd.DataFrame(study_plan)

study_plan

In [ ]:
daily_study_hours = study_plan.groupby("date")["hours"].sum().reset_index()

daily_study_hours["status"] = daily_study_hours["hours"].apply(lambda x: "Overloaded" if x > 2 else "OK")

daily_study_hours

In [ ]:
plt.figure(figsize=(12, 5))
plt.bar(daily_study_hours["date"].dt.strftime("%Y-%m-%d"), daily_study_hours["hours"])
plt.xlabel("Date")
plt.ylabel("Planned Study Hours")
plt.title("Daily Planned Study Hours")
plt.xticks(rotation=45)
plt.show()

In [ ]:
study_plan_output = study_plan.copy()

study_plan_output["date"] = study_plan_output["date"].dt.strftime("%Y-%m-%d")

study_plan_output.to_csv("study_plan_output.csv", index=False)

study_plan_output

In [ ]:
courses_clean.to_csv("cleaned_course_data.csv", index=False)

courses_clean.head()

## Results

The project produced three main results.

First, the course data was cleaned and transformed. The course time column was parsed into weekday and class period columns.

Second, the project created visualizations to show course distribution by weekday and course type. These charts help students understand when most courses are offered.

Third, the project included two small tools: a course conflict checker and an exam-aware study planner. The conflict checker identifies overlapping selected courses. The study planner creates review tasks for the seven days before each exam.

## Conclusion

This project shows how Python can be used to solve a practical student problem. By using Pandas, course data can be cleaned, organized, and analyzed. The project also uses simple rules to check course conflicts and generate exam review tasks.

The system is still a simple prototype. In the future, it could be improved by allowing users to enter their own courses, assignments, exams, and available study time. It could also be turned into a small web app.